In [ ]:
import cv2
import numpy as np
import customtkinter as ctk
from PIL import Image, ImageTk
from tkinter import filedialog
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

class SmartImageApp(ctk.CTk):
    def __init__(self):
        super().__init__()

        # --- Window Configuration ---
        self.title("DIP PRO: Integrated Suite | Noor-ul-ain")
        self.geometry("1400x900")
        ctk.set_appearance_mode("dark")
        ctk.set_default_color_theme("blue")

        # --- State Management ---
        self.original_img = None
        self.gray_img = None
        self.final_result = None

        # --- Main Layout ---
        self.grid_columnconfigure(1, weight=1)
        self.grid_rowconfigure(0, weight=1)

        # 1. SIDEBAR
        self.sidebar = ctk.CTkFrame(self, width=220, corner_radius=0)
        self.sidebar.grid(row=0, column=0, sticky="nsew")
        
        ctk.CTkLabel(self.sidebar, text="DIP ENGINE", font=("Orbitron", 22, "bold")).pack(pady=30)
        
        self.load_btn = ctk.CTkButton(self.sidebar, text="UPLOAD IMAGE", command=self.load_image, height=40)
        self.load_btn.pack(pady=10, padx=20)
        
        self.save_btn = ctk.CTkButton(self.sidebar, text="SAVE RESULT", border_width=2, 
                                     fg_color="transparent", command=self.save_result, height=40)
        self.save_btn.pack(pady=10, padx=20)

        # 2. MAIN CONTENT
        self.tabview = ctk.CTkTabview(self, corner_radius=15)
        self.tabview.grid(row=0, column=1, padx=20, pady=20, sticky="nsew")
        
        tab_list = ["1. Acquisition", "2. Sample/Quant", "3. Geometric", "4. Intensity", "5. Histogram", "6. SMART SYSTEM"]
        for name in tab_list:
            self.tabview.add(name)

        self.setup_acquisition_tab()
        self.setup_sampling_tab()
        self.setup_geometric_tab()
        self.setup_intensity_tab()
        self.setup_histogram_tab()
        self.setup_final_tab()

    # ==========================================
    # HELPER: UPDATE INTEGRATED IMAGE DISPLAY
    # ==========================================
    def update_display(self, cv_img, label_widget):
        if cv_img is None: return
        # Convert BGR/Gray to RGB for Tkinter
        rgb = cv2.cvtColor(cv_img, cv2.COLOR_GRAY2RGB) if len(cv_img.shape) == 2 else cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB)
        
        pil_img = Image.fromarray(rgb)
        
        # Determine Resize Ratio to fit widget (Limit to 1000px wide)
        w_limit, h_limit = 1000, 500
        pil_img.thumbnail((w_limit, h_limit))
        
        tk_img = ImageTk.PhotoImage(pil_img)
        label_widget.configure(image=tk_img, text="")
        label_widget.image = tk_img

    # ==========================================
    # LAB 01: ACQUISITION
    # ==========================================
    def setup_acquisition_tab(self):
        tab = self.tabview.tab("1. Acquisition")
        self.info_label = ctk.CTkLabel(tab, text="Metadata: No Image Loaded", font=("Consolas", 14))
        self.info_label.pack(pady=10)
        self.matrix_box = ctk.CTkTextbox(tab, width=600, height=200, font=("Consolas", 12), fg_color="#1a1a1a")
        self.matrix_box.pack(pady=10)

    def load_image(self):
        path = filedialog.askopenfilename()
        if path:
            self.original_img = cv2.imread(path)
            self.gray_img = cv2.cvtColor(self.original_img, cv2.COLOR_BGR2GRAY)
            info = f"RESOLUTION: {self.gray_img.shape} | DTYPE: {self.gray_img.dtype}"
            self.info_label.configure(text=info, text_color="#28a745")
            self.matrix_box.delete("1.0", "end")
            self.matrix_box.insert("1.0", f"Pixel Matrix (10x10):\n{self.gray_img[:10, :10]}")

    # ==========================================
    # LAB 02: SAMPLING & QUANTIZATION
    # ==========================================
    def setup_sampling_tab(self):
        tab = self.tabview.tab("2. Sample/Quant")
        f = ctk.CTkFrame(tab, fg_color="transparent")
        f.pack(pady=10)
        ctk.CTkButton(f, text="RUN SCALES", command=self.run_sampling).grid(row=0, column=0, padx=10)
        ctk.CTkButton(f, text="RUN BIT-DEPTH", command=self.run_quant).grid(row=0, column=1, padx=10)
        
        self.sq_display = ctk.CTkLabel(tab, text="Analysis Viewport", fg_color="#1a1a1a", corner_radius=10)
        self.sq_display.pack(pady=20, fill="both", expand=True)

    def run_sampling(self):
        if self.gray_img is None: return
        scales = [0.25, 0.5, 1.0, 2.0]
        previews = []
        h, w = self.gray_img.shape
        for s in scales:
            res = cv2.resize(self.gray_img, (int(w*s), int(h*s)))
            up = cv2.resize(res, (w, h), interpolation=cv2.INTER_NEAREST)
            cv2.putText(up, f"{s}x", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255), 3)
            previews.append(up)
        self.update_display(np.hstack(previews), self.sq_display)

    def run_quant(self):
        if self.gray_img is None: return
        bits = [8, 4, 2]
        previews = []
        for b in bits:
            factor = 256 // (2**b)
            q = (self.gray_img // factor) * factor
            cv2.putText(q, f"{b}-Bit", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255), 3)
            previews.append(q)
        self.update_display(np.hstack(previews), self.sq_display)

    # ==========================================
    # LAB 03: GEOMETRIC (NOW INTEGRATED)
    # ==========================================
    def setup_geometric_tab(self):
        tab = self.tabview.tab("3. Geometric")
        self.rot_label = ctk.CTkLabel(tab, text="Rotate Control: 0°")
        self.rot_label.pack(pady=10)
        
        self.geo_slider = ctk.CTkSlider(tab, from_=0, to=180, command=self.apply_rotation)
        self.geo_slider.pack(padx=100, fill="x")
        
        ctk.CTkButton(tab, text="Apply Shear", command=self.apply_shear).pack(pady=20)
        
        self.geo_display = ctk.CTkLabel(tab, text="", fg_color="#1a1a1a", corner_radius=10)
        self.geo_display.pack(pady=10, fill="both", expand=True)

    def apply_rotation(self, val):
        if self.gray_img is None: return
        angle = int(float(val))
        self.rot_label.configure(text=f"Rotate: {angle}°")
        M = cv2.getRotationMatrix2D((self.gray_img.shape[1]//2, self.gray_img.shape[0]//2), angle, 1)
        res = cv2.warpAffine(self.gray_img, M, (self.gray_img.shape[1], self.gray_img.shape[0]))
        self.update_display(res, self.geo_display)

    def apply_shear(self):
        if self.gray_img is None: return
        M = np.float32([[1, 0.2, 0], [0, 1, 0]])
        res = cv2.warpAffine(self.gray_img, M, (int(self.gray_img.shape[1]*1.2), self.gray_img.shape[0]))
        self.update_display(res, self.geo_display)

    # ==========================================
    # LAB 04: INTENSITY (NOW INTEGRATED)
    # ==========================================
    def setup_intensity_tab(self):
        tab = self.tabview.tab("4. Intensity")
        f = ctk.CTkFrame(tab, fg_color="transparent")
        f.pack(pady=10)
        
        ctk.CTkButton(f, text="Negative", command=self.apply_negative).grid(row=0, column=0, padx=10)
        ctk.CTkButton(f, text="Log Transform", command=self.apply_log).grid(row=0, column=1, padx=10)
        
        self.g_slider_val = ctk.CTkSlider(tab, from_=0.1, to=3.0)
        self.g_slider_val.pack(pady=10, padx=100, fill="x")
        ctk.CTkButton(tab, text="Apply Gamma", command=self.apply_gamma).pack(pady=5)
        
        self.int_display = ctk.CTkLabel(tab, text="", fg_color="#1a1a1a", corner_radius=10)
        self.int_display.pack(pady=10, fill="both", expand=True)

    def apply_negative(self):
        if self.gray_img is None: return
        self.update_display(255 - self.gray_img, self.int_display)

    def apply_log(self):
        if self.gray_img is None: return
        c = 255 / np.log(1 + np.max(self.gray_img))
        log_img = (c * (np.log(1 + self.gray_img))).astype(np.uint8)
        self.update_display(log_img, self.int_display)

    def apply_gamma(self):
        if self.gray_img is None: return
        g = self.g_slider_val.get()
        lut = np.array([((i/255.0)**(1.0/g))*255 for i in np.arange(0,256)]).astype("uint8")
        self.update_display(cv2.LUT(self.gray_img, lut), self.int_display)

    # ==========================================
    # LAB 05: HISTOGRAM
    # ==========================================
    def setup_histogram_tab(self):
        tab = self.tabview.tab("5. Histogram")
        ctk.CTkButton(tab, text="Compare Histograms", command=self.run_hist).pack(pady=10)
        self.hist_f = ctk.CTkFrame(tab, fg_color="#1a1a1a")
        self.hist_f.pack(fill="both", expand=True, padx=20, pady=10)

    def run_hist(self):
        if self.gray_img is None: return
        for w in self.hist_f.winfo_children(): w.destroy()
        fig, ax = plt.subplots(1, 2, figsize=(8, 3), facecolor='#1a1a1a')
        for i, (t, d) in enumerate([("Original", self.gray_img), ("Equalized", cv2.equalizeHist(self.gray_img))]):
            ax[i].hist(d.ravel(), 256, [0, 256], color='#1f6aa5')
            ax[i].set_title(t, color='white'); ax[i].tick_params(colors='white')
        canvas = FigureCanvasTkAgg(fig, master=self.hist_f)
        canvas.draw(); canvas.get_tk_widget().pack(fill="both", expand=True)

    # ==========================================
    # LAB 06: SMART SYSTEM
    # ==========================================
    def setup_final_tab(self):
        tab = self.tabview.tab("6. SMART SYSTEM")
        ctk.CTkButton(tab, text="EXECUTE SMART PIPELINE", height=50, fg_color="#28a745", 
                      command=self.run_smart).pack(pady=20)
        self.smart_display = ctk.CTkLabel(tab, text="", fg_color="#1a1a1a", corner_radius=10)
        self.smart_display.pack(pady=10, fill="both", expand=True)

    def run_smart(self):
        if self.gray_img is None: return
        lut = np.array([((i/255.0)**(1.0/1.6))*255 for i in np.arange(0,256)]).astype("uint8")
        self.final_result = cv2.equalizeHist(cv2.LUT(self.gray_img, lut))
        self.update_display(np.hstack([self.gray_img, self.final_result]), self.smart_display)

    def save_result(self):
        if self.final_result is not None:
            path = filedialog.asksaveasfilename(defaultextension=".jpg")
            if path: cv2.imwrite(path, self.final_result)

if __name__ == "__main__":
    app = SmartImageApp()
    app.mainloop()